In [ ]:
!pip install langchain langchain-core langchain-openai openai langchain-anthropic langchain-google-genai google-generativeai langchain-huggingface transformers huggingface-hub python-dotenv numpy scikit-learn

In [ ]:
!pip install grandalf

#**Chains**
>*sequential chains, parellel chains, conditional chains*

###Sequential Chains
>These are already used in previous notebooks

###Parallel Chains
> *These are used to do things parallel*

In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel
from google.colab import userdata
huggingface_api_key = userdata.get('HUGGINGFACEHUB_ACCESS_TOKEN')

parser = StrOutputParser()
llm = HuggingFaceEndpoint(huggingfacehub_api_token=huggingface_api_key, repo_id="meta-llama/Llama-3.1-8B-Instruct", task = "text-generation")
model = ChatHuggingFace(llm=llm)

template1 = PromptTemplate(template="generate me a 5 line description of the topic {topic}", input_variables=["notes"])
template2 = PromptTemplate(template="generate me 5 one word MCQ question on the topic {topic}", input_variables=["notes"])
template3 = PromptTemplate(template="combine me the summary {summary} and the quiz {quiz}", input_variables=["summary", "quiz"])

parellel_chain = RunnableParallel({
    "summary": template1 | model | parser,
    "quiz": template2 | model | parser
})

combiner_chain = parellel_chain | template3 | model | parser
print(combiner_chain.invoke({"topic": "AI"}))

combiner_chain.get_graph().print_ascii()

Here's a combined summary of the topic Artificial Intelligence (AI) and the quiz:

**Artificial Intelligence (AI) Summary**

Artificial Intelligence (AI) refers to the development of computer systems capable of performing tasks that typically require human intelligence. AI involves the creation of algorithms and models that enable machines to learn, reason, and interact with their environment. This technology has numerous applications in various fields, including healthcare, finance, transportation, and education. AI systems can process vast amounts of data, recognize patterns, and make decisions based on that information. By advancing AI research, scientists hope to create more autonomous, intelligent, and efficient systems that can improve our daily lives.

**Quiz: AI Concepts**

1. **What is Cognitive AI?**
   Answer: Cognitive AI is a type of AI that simulates human thought processes. (D) Cognitive

2. **What is Rule-Based AI?**
   Answer: Rule-Based AI is an AI that is controlled 

###Conditional Chains
> *These are used to run a chain on a condition/function being true*

In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnableBranch, RunnableLambda
from pydantic import BaseModel, Field
from typing import Literal
from google.colab import userdata
huggingface_api_key = userdata.get('HUGGINGFACEHUB_ACCESS_TOKEN')

class SentimentClass(BaseModel):
  sentiment: Literal["positive", "negative"] = Field(description="sentiment of the text as positive or negative")

parser1 = PydanticOutputParser(pydantic_object=SentimentClass)
parser2 = StrOutputParser()

llm = HuggingFaceEndpoint(huggingfacehub_api_token=huggingface_api_key, repo_id="meta-llama/Llama-3.1-8B-Instruct", task = "text-generation")
model = ChatHuggingFace(llm=llm)

template = PromptTemplate(template="Identify the sentiment of following text {text} \n {format_instruction}", input_variables=["text"], partial_variables={"format_instruction": parser1.get_format_instructions()})
template1 = PromptTemplate(template="Generate Feedback for the positive review {review}", input_variables=["review"])
template2 = PromptTemplate(template="Generate a mail for the negative review {review}", input_variables=["review"])

parent_chain = template | model | parser1
conditional_chain = RunnableBranch(
    (lambda x: x.sentiment == "positive", template1 | model | parser2),
    (lambda x: x.sentiment == "negative", template2 | model | parser2),
    RunnableLambda(lambda x: f"Could not determine sentiment of text")
)

final_chain = parent_chain | conditional_chain
print(final_chain.invoke({"text": "I like this product"}))

final_chain.get_graph().print_ascii()

Here's some potential feedback based on a positive review with a sentiment of 'positive':

**Thank you for your wonderful review!**

We're thrilled to hear that you had a great experience with [product/service]. Our team works hard to ensure that every customer is satisfied, and it's reviews like yours that make all our efforts worthwhile.

**Specific Feedback:**

- Your kind words about our [product/service] are music to our ears. We're glad you found it [insert positive adjective, e.g. "easy to use" or "high-quality"].
- We appreciate the time you took to share your experience with others. Your review will help us continue to improve and provide the best possible service to our customers.
- Thank you for choosing [company name]! We're grateful for customers like you who believe in our mission and values.

**Future Improvements:**

- We're always looking for ways to improve our [product/service]. If there's anything you think we could do better, please don't hesitate to let us know. Y

#**Runnables**
> *RunnableSequence, RunnableParallel, RunnablePassthrough, RunnableLambda, RunnableBranch*

###RunnableSequence

In [ ]:
from langchain_core.runnables import RunnableSequence

chain = RunnableSequence(template, model, parser) # chain = template | model | parser

###RunnableParallel

In [ ]:
from langchain_core.runnables import RunnableParallel
parellel_chain = RunnableParallel({
    "summary": template1 | model | parser,
    "quiz": template2 | model | parser
})

'''you can also use'''
# parellel_chain = {
#     "summary": template1 | model | parser,
#     "quiz": template2 | model | parser
# }

combiner_chain = parellel_chain | template3 | model | parser

###RunnablePassthrough
> *returns input as output without modifications*

In [ ]:
# usecase, to display output of a node middle of the chain
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from google.colab import userdata
huggingface_api_key = userdata.get('HUGGINGFACEHUB_ACCESS_TOKEN')

parser = StrOutputParser()
llm = HuggingFaceEndpoint(huggingfacehub_api_token=huggingface_api_key, repo_id="meta-llama/Llama-3.1-8B-Instruct", task = "text-generation")
model = ChatHuggingFace(llm=llm)

template1 = PromptTemplate(template="write a joke about {topic}", input_variables=["topic"])
template2 = PromptTemplate(template="explain the joke {topic}", input_variables=["topic"])
parent_chain = template1 | model | parser
parallel_chain = {
    "res1": RunnablePassthrough(),
    "chain2": template2 | model | parser
}

final_chain = parent_chain | parallel_chain
print(final_chain.invoke({"topic": "India"}))

{'res1': 'Why did the Indian elephant quit his job?\n\nBecause he was tired of working for peanuts.', 'chain2': "This joke relies on a play on words and uses a form of wordplay known as a pun. \n\nThe phrase 'working for peanuts' has a dual meaning here: \n\n1. An elephant working on a farm or in a circus might be tasked with picking peanuts, which are a type of nut. Therefore, an elephant working for peanuts might be literally collecting or handling these nuts.\n\n2. More commonly, 'working for peanuts' is an idiomatic expression that means receiving very low pay for one's work. In this context, the elephant is quitting his job because he is dissatisfied with the low wage he is receiving.\n\nThe joke works by using the literal meaning of 'peanuts' (the nuts) to create the punchline, which is unexpected but also makes sense in the context of the phrase. The humor comes from the unexpected twist on the usual idiomatic expression, and the clever use of wordplay."}


###RunnableLambda
> *enables to convert normal python funcitons to runnable functions*

In [ ]:
#syntax
RunnableLambda(function_name)

###RunnableBranch
> *Chains with conditions*

In [ ]:
# syntax
conditional_chain = RunnableBranch(
    (lambda x: x.sentiment == "positive", template1 | model | parser2),
    (lambda x: x.sentiment == "negative", template2 | model | parser2),
    RunnableLambda(lambda x: f"Could not determine sentiment of text")
)